In [5]:
!apt-get update
!apt-get install -y tesseract-ocr
!apt-get install -y tesseract-ocr-mar
!pip install pytesseract pdf2image opencv-python-headless
!pip install pdfplumber

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading

In [10]:
!apt-get update
!apt-get install -y tesseract-ocr
!apt-get install -y tesseract-ocr-mar

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading

In [17]:
import os
import re
import cv2
import numpy as np
import pandas as pd
import pdfplumber
import pytesseract
import pdf2image
from PIL import Image
from google.colab import files
import ipywidgets as widgets
from IPython.display import display, clear_output

# Set pytesseract path (if needed)
pytesseract.pytesseract.tesseract_cmd = r'/usr/bin/tesseract'


In [18]:

# Set custom tesseract configurations with better parameters for Marathi+English
custom_config = r'--oem 3 --psm 6 -l mar+eng -c preserve_interword_spaces=1'

# Function to extract table from PDF using pdfplumber (for text-based PDFs)
def extract_table_from_pdf(pdf_path):
    try:
        with pdfplumber.open(pdf_path) as pdf:
            all_data = []
            for page in pdf.pages:
                # Extract tables from each page
                tables = page.extract_tables()
                for table in tables:
                    # Combine all table data
                    all_data.extend(table)
            return all_data
    except Exception as e:
        print(f"Error using pdfplumber: {e}")
        return None

# Function to convert PDF to images with higher DPI for better quality
def convert_pdf_to_images(pdf_path, dpi=400):
    try:
        return pdf2image.convert_from_path(pdf_path, dpi=dpi)
    except Exception as e:
        print(f"Error converting PDF to images: {e}")
        return None

# Enhanced image preprocessing for better OCR results
def enhance_image(image):
    # Convert PIL Image to numpy array if needed
    if isinstance(image, Image.Image):
        image = np.array(image)

    # Convert to grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Apply CLAHE for improved contrast
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    enhanced = clahe.apply(gray)

    # Try different thresholding approach
    # First, try Otsu's method
    _, otsu = cv2.threshold(enhanced, 0, 255, cv2.THRESH_BINARY+cv2.THRESH_OTSU)

    # Also try adaptive thresholding
    adaptive = cv2.adaptiveThreshold(enhanced, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                  cv2.THRESH_BINARY, 11, 2)

    # Check which one has better contrast (higher standard deviation usually means better separation)
    if np.std(otsu) > np.std(adaptive):
        binary = otsu
    else:
        binary = adaptive

    # Remove noise with morphological operations
    kernel = np.ones((2, 2), np.uint8)
    opening = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)

    # Apply dilation to make text clearer
    kernel = np.ones((1, 1), np.uint8)
    dilated = cv2.dilate(opening, kernel, iterations=1)

    # Save debug image
    cv2.imwrite("/content/enhanced_image.png", dilated)

    return dilated

# Improved function for table structure detection using Hough Line Transform
def detect_table_structure(image):
    # Convert PIL Image to numpy array if needed
    if isinstance(image, Image.Image):
        image = np.array(image)

    # Convert to grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Apply thresholding
    _, binary = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY_INV)

    # Edge detection
    edges = cv2.Canny(binary, 50, 150, apertureSize=3)

    # Use Hough Line Transform to detect lines
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=100, minLineLength=100, maxLineGap=10)

    # Create blank image to draw lines
    line_image = np.zeros_like(gray)

    if lines is not None:
        for line in lines:
            x1, y1, x2, y2 = line[0]
            cv2.line(line_image, (x1, y1), (x2, y2), 255, 2)

    # Find horizontal lines
    horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (50, 1))
    horizontal_lines = cv2.morphologyEx(binary, cv2.MORPH_OPEN, horizontal_kernel)

    # Find vertical lines
    vertical_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 50))
    vertical_lines = cv2.morphologyEx(binary, cv2.MORPH_OPEN, vertical_kernel)

    # Combine horizontal and vertical lines
    table_structure = cv2.add(horizontal_lines, vertical_lines)

    # Combine with Hough lines for better structure detection
    table_structure = cv2.add(table_structure, line_image)

    # Save debug image
    cv2.imwrite("/content/table_structure.png", table_structure)

    # Find contours of the grid
    contours, _ = cv2.findContours(table_structure, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

    return contours, table_structure

# Improved function to extract cells from table structure
def extract_cells_from_table(image, table_structure):
    # Create a copy to draw on for debugging
    debug_img = image.copy() if isinstance(image, np.ndarray) else np.array(image).copy()

    # Find contour intersections to identify cells
    contours, _ = cv2.findContours(table_structure, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

    # Get bounding rectangles for each contour
    cells = []
    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)
        # Filter out very small or very large rectangles
        if w > 20 and h > 20 and w < image.shape[1] - 10 and h < image.shape[0] - 10:
            cells.append((x, y, w, h))
            # Draw rectangle on debug image
            cv2.rectangle(debug_img, (x, y), (x+w, y+h), (0, 255, 0), 2)

    # Save debug image with cell boundaries
    cv2.imwrite("/content/cell_boundaries.png", debug_img)

    # If no cells found, try alternative approach - grid-based division
    if len(cells) < 5:  # Arbitrary threshold - if too few cells found
        print("Few cells detected, trying grid-based approach...")
        # Find potential row divisions (horizontal lines)
        row_projections = np.sum(table_structure, axis=1)
        row_peaks = np.where(row_projections > np.mean(row_projections))[0]

        # Find potential column divisions (vertical lines)
        col_projections = np.sum(table_structure, axis=0)
        col_peaks = np.where(col_projections > np.mean(col_projections))[0]

        # Create cells based on grid intersections
        cells = []
        for i in range(len(row_peaks)-1):
            for j in range(len(col_peaks)-1):
                x = col_peaks[j]
                y = row_peaks[i]
                w = col_peaks[j+1] - col_peaks[j]
                h = row_peaks[i+1] - row_peaks[i]
                if w > 20 and h > 20:
                    cells.append((x, y, w, h))

    # Sort cells by y-coordinate (top to bottom)
    cells.sort(key=lambda cell: cell[1])

    # Group cells into rows based on y-coordinate proximity
    rows = []
    current_row = []
    current_y = cells[0][1] if cells else 0

    for cell in cells:
        x, y, w, h = cell
        # If this cell is on a new row
        if y > current_y + 20:  # Threshold for new row
            if current_row:
                # Sort the row by x-coordinate (left to right)
                current_row.sort(key=lambda cell: cell[0])
                rows.append(current_row)
            current_row = [cell]
            current_y = y
        else:
            current_row.append(cell)

    # Add the last row
    if current_row:
        current_row.sort(key=lambda cell: cell[0])
        rows.append(current_row)

    return rows

# Improved function for direct table detection without relying on grid lines
def detect_table_area(image):
    # Convert PIL Image to numpy array if needed
    if isinstance(image, Image.Image):
        image = np.array(image)

    # Convert to grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Apply thresholding
    _, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY_INV)

    # Find contours
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Find the largest contour which should be the table
    max_area = 0
    table_contour = None
    for contour in contours:
        area = cv2.contourArea(contour)
        if area > max_area:
            max_area = area
            table_contour = contour

    if table_contour is not None:
        x, y, w, h = cv2.boundingRect(table_contour)
        return (x, y, w, h)

    return None

# Improved function to extract table from image with better OCR handling
def extract_table_from_image(image, lang='mar+eng'):
    # Make a copy of the image for processing
    img_copy = image.copy() if isinstance(image, np.ndarray) else np.array(image)

    # Enhance image for better OCR
    enhanced = enhance_image(img_copy)

    # Create a debug image
    debug_img = img_copy.copy()

    # Try multiple approaches and use the best result
    all_results = []
    confidence_scores = []

    # Method 1: Try pytesseract's built-in table detection with PSM 6
    try:
        print("Trying Method 1: Tesseract's built-in table detection...")
        # Use Tesseract's built-in table detection (page segmentation mode 6)
        table_data = pytesseract.image_to_data(
            enhanced,
            lang=lang,
            config='--oem 3 --psm 6',
            output_type=pytesseract.Output.DATAFRAME
        )

        # Filter out empty or low-confidence text
        table_data = table_data[table_data['conf'] > 40]  # Increased confidence threshold
        table_data = table_data[table_data['text'].str.strip() != '']

        # Group by block and line to reconstruct table structure
        if not table_data.empty:
            rows = []
            for block_num, block_data in table_data.groupby('block_num'):
                for line_num, line_data in block_data.groupby('line_num'):
                    # Sort words by position in line
                    sorted_words = line_data.sort_values('left')
                    row_text = ' '.join(sorted_words['text'].tolist())

                    # Check if this is likely a table row
                    if ',' in row_text or '\t' in row_text or len(sorted_words) > 3:
                        # Split by commas, tabs, or multiple spaces
                        cells = re.split(r'[,\t]|\s{3,}', row_text)
                        cells = [cell.strip() for cell in cells if cell.strip()]
                        if cells:
                            rows.append(cells)

            if rows:
                all_results.append(rows)
                # Calculate a confidence score based on regularity of rows and columns
                avg_cols = sum(len(row) for row in rows) / len(rows) if rows else 0
                col_variance = sum((len(row) - avg_cols) ** 2 for row in rows) / len(rows) if rows else 999
                confidence_scores.append(1000 / (1 + col_variance))  # Higher score for more consistent columns
    except Exception as e:
        print(f"Method 1 failed: {e}")

    # Method 2: Try with PSM 4 (fully automatic page segmentation with OCR)
    try:
        print("Trying Method 2: Fully automatic page segmentation...")
        all_text = pytesseract.image_to_string(
            enhanced,
            lang=lang,
            config='--oem 3 --psm 4 -c preserve_interword_spaces=1'
        )

        # Split into rows
        lines = all_text.split('\n')
        lines = [line.strip() for line in lines if line.strip()]

        # Try to identify table rows
        rows = []
        for line in lines:
            # Split by comma, tab, or multiple spaces
            cells = re.split(r'[,\t]|\s{3,}', line)
            cells = [cell.strip() for cell in cells if cell.strip()]

            if len(cells) > 1:  # Only consider as table row if multiple cells
                rows.append(cells)

        if rows:
            all_results.append(rows)
            # Calculate confidence based on number of rows and regularity
            avg_cols = sum(len(row) for row in rows) / len(rows) if rows else 0
            col_variance = sum((len(row) - avg_cols) ** 2 for row in rows) / len(rows) if rows else 999
            confidence_scores.append(800 / (1 + col_variance))  # Slightly lower base confidence than method 1
    except Exception as e:
        print(f"Method 2 failed: {e}")

    # Method 3: Use OpenCV to detect the table structure
    try:
        print("Trying Method 3: OpenCV table structure detection...")
        # Detect table structure using contours
        contours, table_structure = detect_table_structure(img_copy)

        # Extract cells from the table structure
        cell_rows = extract_cells_from_table(img_copy, table_structure)

        # Extract text from each cell
        rows = []
        for row in cell_rows:
            row_data = []
            for x, y, w, h in row:
                # Draw the cell on debug image
                cv2.rectangle(debug_img, (x, y), (x+w, y+h), (0, 255, 0), 2)

                # Extract the cell image with padding
                cell_img = enhanced[max(0, y-5):min(enhanced.shape[0], y+h+5),
                                   max(0, x-5):min(enhanced.shape[1], x+w+5)]

                if cell_img.size == 0:
                    row_data.append("")
                    continue

                # Try multiple PSM modes and pick the one with highest confidence
                best_cell_text = ""
                best_conf = 0

                for psm in [7, 8, 6]:  # Single line, single word, and uniform block
                    try:
                        cell_data = pytesseract.image_to_data(
                            cell_img,
                            lang=lang,
                            config=f'--oem 3 --psm {psm} -c preserve_interword_spaces=1',
                            output_type=pytesseract.Output.DICT
                        )

                        # Calculate average confidence
                        confidences = [conf for conf in cell_data['conf'] if conf != -1]
                        avg_conf = sum(confidences) / len(confidences) if confidences else 0

                        cell_text = ' '.join([word for word, conf in
                                             zip(cell_data['text'], cell_data['conf'])
                                             if conf > 30 and word.strip()])

                        if avg_conf > best_conf and cell_text.strip():
                            best_conf = avg_conf
                            best_cell_text = cell_text
                    except Exception:
                        continue

                # Clean up text by removing non-printable characters
                best_cell_text = re.sub(r'[^\x00-\x7F\u0900-\u097F]+', '', best_cell_text).strip()
                row_data.append(best_cell_text)

            if any(row_data):  # Only add non-empty rows
                rows.append(row_data)

        if rows:
            all_results.append(rows)
            # Calculate confidence based on filled cells and regularity
            filled_ratio = sum(1 for row in rows for cell in row if cell.strip()) / sum(len(row) for row in rows) if sum(len(row) for row in rows) > 0 else 0
            avg_cols = sum(len(row) for row in rows) / len(rows) if rows else 0
            col_variance = sum((len(row) - avg_cols) ** 2 for row in rows) / len(rows) if rows else 999
            confidence_scores.append(900 * filled_ratio / (1 + col_variance))
    except Exception as e:
        print(f"Method 3 failed: {e}")

    # Method 4: Regular line-by-line OCR with whitespace analysis
    try:
        print("Trying Method 4: Line-by-line OCR with whitespace analysis...")
        all_text = pytesseract.image_to_string(
            enhanced,
            lang=lang,
            config='--oem 3 --psm 6 -c preserve_interword_spaces=1'
        )

        # Split into rows
        lines = all_text.split('\n')
        lines = [line.strip() for line in lines if line.strip()]

        # Try to identify table structure by finding consistent whitespace patterns
        rows = []
        for line in lines:
            # Look for repeating patterns of multiple spaces
            spaces = [m.start() for m in re.finditer(r'\s{2,}', line)]

            if spaces:
                # Use spaces as column separators
                cells = []
                start = 0
                for space in spaces:
                    cells.append(line[start:space].strip())
                    start = space + 1
                cells.append(line[start:].strip())  # Add the last cell

                cells = [cell for cell in cells if cell]  # Remove empty cells
                if len(cells) > 1:  # Only add if multiple cells
                    rows.append(cells)
            else:
                # Try Devanagari specific patterns
                # In Marathi text, certain characters often indicate field boundaries
                cells = re.split(r'[,।\|]', line)
                cells = [cell.strip() for cell in cells if cell.strip()]
                if len(cells) > 1:
                    rows.append(cells)

        if rows:
            all_results.append(rows)
            # Calculate simple confidence based on number of rows and columns
            confidence_scores.append(min(700, len(rows) * 100))  # Simple metric based on number of rows
    except Exception as e:
        print(f"Method 4 failed: {e}")

    # Save debug image
    cv2.imwrite("/content/cell_debug.png", debug_img)

    # Select the best result based on confidence scores
    if all_results:
        best_idx = confidence_scores.index(max(confidence_scores))
        print(f"Selected Method {best_idx + 1} with confidence score: {confidence_scores[best_idx]}")
        return all_results[best_idx]

    # If all methods fail, return empty list
    print("All methods failed to extract meaningful table data.")
    return []

# Main function to process document and extract tables with better handling
def process_document(file_path, lang='mar+eng'):
    file_ext = os.path.splitext(file_path)[1].lower()

    print(f"Processing file: {file_path} with extension {file_ext}")
    print(f"Selected language: {lang}")

    if file_ext == '.pdf':
        # First try pdfplumber for text-based PDFs
        print("Trying pdfplumber for text-based PDF extraction...")
        table_data = extract_table_from_pdf(file_path)

        # If no tables found or error, try OCR approach
        if not table_data:
            print("No tables found with pdfplumber, trying OCR approach...")
            # Convert PDF to images at higher resolution
            images = convert_pdf_to_images(file_path, dpi=400)

            if images:
                all_table_data = []
                for i, img in enumerate(images):
                    print(f"Processing page {i+1} of {len(images)}...")
                    # Extract tables from image
                    img_table_data = extract_table_from_image(img, lang=lang)
                    if img_table_data:
                        print(f"Found table data on page {i+1} with {len(img_table_data)} rows")
                        all_table_data.extend(img_table_data)
                    else:
                        print(f"No table data found on page {i+1}")
                table_data = all_table_data
            else:
                print("Failed to convert PDF to images")
                table_data = []
        return table_data
    else:  # Image file
        print("Processing as image file...")
        try:
            # Try with PIL first
            image = Image.open(file_path)
            print(f"Image opened with PIL: {image.size}")
            # Extract tables from image
            table_data = extract_table_from_image(image, lang=lang)
            return table_data
        except Exception as e:
            print(f"Error processing image with PIL: {e}")
            try:
                # Try with OpenCV as backup
                image = cv2.imread(file_path)
                if image is None:
                    print("Failed to open image file with OpenCV")
                    return []
                print(f"Image opened with OpenCV: {image.shape}")
                # Extract tables from image
                table_data = extract_table_from_image(image, lang=lang)
                return table_data
            except Exception as e:
                print(f"All image processing methods failed: {e}")
                return []

# Improved post-processing for Marathi text
def post_process_table_data(table_data):
    if not table_data:
        return table_data

    # Common OCR errors in Marathi/Devanagari text
    marathi_corrections = {
        # Common OCR mistakes - update with actual patterns you observe
        'ा': 'ा',      # Fix specific character issues
        'े': 'े',      # Fix vowel signs
        'ि': 'ि',
        '०': '0',      # Number conversions if needed
        '१': '1',
        '२': '2',
        '३': '3',
        '४': '4',
        '५': '5',
        '६': '6',
        '७': '7',
        '८': '8',
        '९': '9',
        # Add more corrections as you observe patterns
    }

    # Clean up and normalize the data
    processed_data = []
    for row in table_data:
        processed_row = []
        for cell in row:
            # Remove CID placeholders and other special characters
            clean_cell = re.sub(r'\(cid:[^\)]+\)', '', cell)

            # Remove non-Devanagari, non-English characters except basic punctuation
            clean_cell = re.sub(r'[^\u0900-\u097F\u0020-\u007E\.,\-\"\'\n]', '', clean_cell)

            # Apply Marathi-specific corrections
            for wrong, correct in marathi_corrections.items():
                clean_cell = clean_cell.replace(wrong, correct)

            # Normalize whitespace
            clean_cell = re.sub(r'\s+', ' ', clean_cell).strip()

            processed_row.append(clean_cell)

        # Only add non-empty rows
        if any(cell for cell in processed_row):
            processed_data.append(processed_row)

    return processed_data

# Improved function to normalize and save table data to CSV
def normalize_and_save_to_csv(table_data, output_path):
    # Post-process the data to clean it up
    table_data = post_process_table_data(table_data)

    if not table_data or len(table_data) == 0:
        print("No table data found!")
        return False

    # Remove empty rows and normalize the table
    normalized_data = [row for row in table_data if any(row)]
    if not normalized_data:
        print("No non-empty data found!")
        return False

    # Print preview of raw data
    print("\nRaw extracted data preview:")
    for i, row in enumerate(normalized_data[:5]):
        print(f"Row {i+1}: {row}")

    # Find maximum number of columns for padding
    max_columns = max(len(row) for row in normalized_data)
    print(f"Detected {max_columns} columns")

    # Check if we need to add headers
    if len(normalized_data) > 1:
        first_row = normalized_data[0]

        # Analyze if first row could be a header
        has_header = len(first_row) == max_columns  # Header should have all columns
        for cell in first_row:
            # If cell is numeric or empty, likely not a header
            if cell.isdigit() or not cell:
                has_header = False
                break

        if has_header:
            print("First row detected as header")
            headers = first_row
            data_rows = normalized_data[1:]
        else:
            print("No header detected, generating default column names")
            headers = [f"Column_{i+1}" for i in range(max_columns)]
            data_rows = normalized_data
    else:
        headers = [f"Column_{i+1}" for i in range(max_columns)]
        data_rows = normalized_data

    # Pad all rows to have the same number of columns
    padded_data = [row + [""] * (max_columns - len(row)) for row in data_rows]

    # Create a DataFrame
    df = pd.DataFrame(padded_data, columns=headers)

    # Save to CSV with proper encoding for Devanagari
    df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"CSV saved to {output_path}")

    # Preview the data
    print("\nPreview of processed data:")
    print(df.head().to_string())

    return True

# Debugging helper function
def save_debug_image(image, name):
    """Save an image for debugging purposes"""
    if isinstance(image, Image.Image):
        image = np.array(image)

    cv2.imwrite(f"/content/debug_{name}.png", image)

# GUI Components
upload_button = widgets.FileUpload(
    accept='.pdf,.jpg,.jpeg,.png',
    multiple=False,
    description='Upload File'
)
lang_dropdown = widgets.Dropdown(
    options=[('Marathi+English', 'mar+eng'), ('Marathi', 'mar'), ('English', 'eng'), ('Hindi+English', 'hin+eng')],
    value='mar+eng',
    description='Language:'
)
output_area = widgets.Output()
download_button = widgets.Button(description="Download CSV", button_style="success")
output_path = "extracted_table.csv"

# Updated file upload handler
def handle_file_upload(change):
    with output_area:
        clear_output()
        if not upload_button.value:
            print("No file uploaded. Please upload a file.")
            return

        try:
            # Get the first file from the uploaded files
            file_key = next(iter(upload_button.value))
            uploaded_file = upload_button.value[file_key]

            # Get content
            content = uploaded_file['content']

            # Try different ways to get the filename
            try:
                file_name = uploaded_file['metadata']['name']
            except KeyError:
                try:
                    file_name = uploaded_file['name']
                except KeyError:
                    # Use the key as the filename with an appropriate extension
                    file_name = f"uploaded_file_{file_key}"
                    # Try to determine file type based on content
                    if content[:4] == b'%PDF':
                        file_name += '.pdf'
                    else:
                        file_name += '.jpg'

            file_path = f'/content/{file_name}'

            print(f"Uploaded file: {file_name}")

            with open(file_path, 'wb') as f:
                f.write(content)

            print(f"Processing {file_name} with language: {lang_dropdown.value}...")
            print("This may take a moment depending on the file size and complexity...")

            # Process document and extract tables
            table_data = process_document(file_path, lang=lang_dropdown.value)

            # Normalize and save to CSV
            success = normalize_and_save_to_csv(table_data, output_path)

            if success:
                print("\nCSV is ready for download. Click the 'Download CSV' button.")
            else:
                print("\nNo valid table data could be extracted. Try adjusting the language or uploading a different file.")
        except Exception as e:
            print(f"Error: {e}")
            import traceback
            traceback.print_exc()

def handle_download():
    try:
        files.download(output_path)
    except Exception as e:
        with output_area:
            print(f"Error downloading file: {e}")

# Attach handlers to widgets
upload_button.observe(handle_file_upload, names='value')
download_button.on_click(lambda b: handle_download())

# Display GUI
display(widgets.VBox([
    widgets.HTML("<h2>Enhanced Marathi Document Table Extractor</h2>"),
    widgets.HTML("<p>Upload PDF or Image with Marathi/Hindi text to extract table data</p>"),
    widgets.HBox([upload_button, lang_dropdown]),
    output_area,
    download_button
]))